# Construction de la base de données — Registres de la Chancellerie royale (AN JJ)

 Ce script transforme trois fichiers CSV sources en quatre tables normalisées importables dans une base de données relationnelle :

 | Table | Description |
 |---|---|
 | `images.csv` | Une ligne par image numérisée |
 | `zones.csv` | Une ligne par zone YOLO détectée |
 | `actes.csv` | Une ligne par acte avec concordances aplaties |
 | `actes_images_zones.csv` | Liaisons acte ↔ image ↔ zone, avec rôles et statuts |

 **Structure valide d'un acte :**
 - `[AC]` — acte complet, zone unique
 - `[AI] + [AF]` — acte initial + final
 - `[AI] + [AM]⁺ + [AF]` — avec une ou plusieurs zones médianes (pages entières)



# Paramètres


In [40]:
import os
import re

corpus = 'JJ200-JJ211'


# Chemins des fichiers sources
INPUT_ZONES  = f"../List-of-zones/Himanis_Seg_Actes_1200pxmin_{corpus}_labelstudio.csv"       # fichier 1 : zones YOLO
INPUT_ACTES  = "../List-of-acts/Acts-JJ96-JJ195_enriched_Locus-20260521.xlsm"                # fichier 2 : liste des actes
INPUT_IMAGES = f"../List-of-images/{corpus}_image_data.csv"    # fichier 3 : images téléchargées

# Dossier de sortie
OUT_DIR = f"{corpus}/output"
os.makedirs(OUT_DIR, exist_ok=True)

# Labels de zones attendus par le modèle d'acte
LABELS_ATTENDUS = {'AC', 'AI', 'AM', 'AF', 'NIA', 'Table'}



def parse_corpus(corpus):
    match = re.match(r'([A-Z]+)(\d+)-([A-Z]+)(\d+)', corpus)
    if not match:
        raise ValueError(f"Format inattendu : {corpus}")
    prefix = match.group(1)
    start, end = int(match.group(2)), int(match.group(4))
    return {f"{prefix}{i}" for i in range(start, end + 1)}

REGISTRES_CIBLES = parse_corpus(corpus)
print(REGISTRES_CIBLES)

# Séparateur CSV des fichiers sources ('\t' = tabulation)
SEP = '\t'


{'JJ207', 'JJ204', 'JJ210', 'JJ200', 'JJ203', 'JJ205', 'JJ206', 'JJ202', 'JJ209', 'JJ208', 'JJ201', 'JJ211'}


# Imports et fonctions utilitaires

In [41]:
!which python
!python --version
!which pip

'which' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.


Python 3.12.0


'which' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.


In [42]:
!pip install openpyxl


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:
import os
import re
import ast
import json
import warnings
import pandas as pd
import numpy as np
from collections import defaultdict


pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)

# %%
def normalize_path(p):
    """Normalise les séparateurs Windows/Unix et renvoie le basename."""
    return os.path.basename(str(p).replace("\\", "/"))


def extract_register(folder_path):
    """
    Extrait le numéro de registre normalisé depuis un chemin.
    'Paris_Archives_Nationales_JJ096' → 'JJ96' (sans zéro initial).
    Cible le DERNIER composant du chemin contenant JJ + chiffres.
    """
    parts = str(folder_path).replace("\\", "/").split("/")
    for part in reversed(parts):
        m = re.search(r'JJ0*(\d+)$', part)
        if m:
            return f"JJ{m.group(1)}"
    return None


def normalize_folio(raw):
    """
    Normalise un label de folio pour jointure.
      '6'        → '6r'
      '12v'      → '12v'
      '1r'       → '1r'
      '103bis'   → '103bisr'
      '103bis v' → '103bisv'
      '103 bis recto' → '103bisr'
      '103 bis verso' → '103bisv'
      'plat supérieur' → 'plat supérieur'
    """
    s = str(raw).strip().lower()

    # Dans normalize_folio, avant toute autre règle :
    if s in ('vacat', 'vacatr', 'vacatv'):
        return None   # sera logué comme "sans folio" → ignoré proprement
    
    if not s or s == 'nan':
        return None

    # Normaliser les variantes textuelles de recto/verso
    s = s.replace('recto', 'r').replace('verso', 'v')

    # Supprimer les espaces autour de 'bis' et avant r/v
    # '103 bis r' → '103bisr', '103 bis v' → '103bisv', '103 bis' → '103bis'
    s = re.sub(r'\s+bis\s*', 'bis', s)  # espaces autour de bis
    s = re.sub(r'\s+([rv])$', r'\1', s) # espace avant r/v final

    # Cas standard : chiffres + optionnel 'bis' + r ou v
    if re.match(r'^\d+(?:bis)?[rv]$', s):
        return s

    # Chiffres + optionnel 'bis' sans suffixe → recto par défaut
    if re.match(r'^\d+(?:bis)?$', s):
        return s + 'r'

    return s


def parse_abs_coords(coord_str):
    """Parse '1,23,3866,6279' → [x, y, w, h] ou None si invalide."""
    try:
        parts = [int(v) for v in str(coord_str).split(',')]
        if len(parts) == 4:
            return parts
    except Exception:
        pass
    return None

def parse_image_stem(image_path):
    stem = normalize_path(image_path).rsplit('.', 1)[0]  # retire .jpg
    parts = stem.split('_')
    volume = parts[-2]           # 'JJ096'
    folio_sort_key = int(parts[-1])  # 100
    return volume, folio_sort_key


def safe_to_int(series):
    return (
        series
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)              # ou dropna selon ton besoin
        .round()
        .astype(int)
    )

# Chargement des fichiers CSV sources

## Table `images.csv`

In [44]:
skipped = {}

with warnings.catch_warnings(record=True) as w_images:
    warnings.simplefilter("always")
    df_images_raw = pd.read_csv(INPUT_IMAGES, sep=',', dtype=str,
                             low_memory=False, on_bad_lines='warn', encoding='utf-8')
    skipped['images'] = len(w_images)

print(f"Images chargées : {len(df_images_raw):>6} lignes  ({skipped['images']} ligne(s) ignorée(s))")

df_images = df_images_raw[[
    'manifestURL', 'canvasId', 'urlImage', 'imageLabel',
    'imageFileName', 'imageWidthAsDownloaded', 'imageHeightAsDownloaded',
    'urlResizedImage', 'ResizedImageWidthAsDownloaded',
    'ResizedImageHeightAsDownloaded', 'folderPath'
]].copy()


df_images[['volume', 'folio_sort_key']] = df_images['imageFileName'].apply(
    lambda p: pd.Series(parse_image_stem(p))
)

df_images['image_id']       = range(1, len(df_images) + 1)
df_images['image_filename'] = df_images['imageFileName'].apply(normalize_path)
df_images[['volume', 'folio_sort_key']] = df_images['image_filename'].apply(
    lambda p: pd.Series(parse_image_stem(p))
    )
df_images['register']       = df_images['folderPath'].apply(extract_register)
df_images['folio_norm']     = df_images['imageLabel'].apply(normalize_folio)

df_images.rename(columns={
    'manifestURL':                    'manifest_url',
    'canvasId':                       'canvas_id',
    'urlImage':                       'url_full',
    'imageLabel':                     'folio_label',
    'imageWidthAsDownloaded':         'width_px',
    'imageHeightAsDownloaded':        'height_px',
    'urlResizedImage':                'url_resized',
    'ResizedImageWidthAsDownloaded':  'resized_width_px',
    'ResizedImageHeightAsDownloaded': 'resized_height_px',
}, inplace=True)

df_images.drop(columns=['imageFileName', 'folderPath'], inplace=True)

IMAGES_COLS = [
    'image_id', 'register', 'image_filename', 'volume', 'folio_sort_key', 
    'folio_label', 'folio_norm',
    'width_px', 'height_px', 'url_full', 'url_resized',
    'resized_width_px', 'resized_height_px', 'manifest_url', 'canvas_id',
]
df_images = df_images[IMAGES_COLS]

df_images.to_csv(os.path.join(OUT_DIR, "images.csv"), index=False, sep=SEP)
print(f"images.csv → {len(df_images)} lignes, {len(df_images.columns)} colonnes")
df_images.head(3)

# %%
# Index (register, folio_norm) → [image_id, ...]  (ordre séquentiel préservé)
img_by_folio = defaultdict(list)
for _, row in df_images.iterrows():
    if pd.notna(row['volume']) and pd.notna(row['folio_sort_key']):
        img_by_folio[(row['volume'], int(row['folio_sort_key']))].append(row['image_id'])

        
# image_id → rang dans le registre (pour navigation séquentielle vers images suivantes)
# On construit un index par registre : register → [image_id ordonné]
images_by_register = defaultdict(list)
for _, row in df_images.iterrows():
    if row['register']:
        images_by_register[row['register']].append(row['image_id'])

print(f"Index img_by_folio    : {len(img_by_folio)} clés (register, folio_norm)")
print(f"Registres distincts   : {list(images_by_register.keys())}")


df_images.head(3)

Images chargées :   3950 lignes  (0 ligne(s) ignorée(s))
images.csv → 3950 lignes, 15 colonnes
Index img_by_folio    : 3950 clés (register, folio_norm)
Registres distincts   : ['JJ200', 'JJ201', 'JJ202', 'JJ203', 'JJ204', 'JJ205', 'JJ206', 'JJ207', 'JJ208', 'JJ209', 'JJ210', 'JJ211']


,image_id,register,image_filename,volume,folio_sort_key,folio_label,folio_norm,width_px,height_px,url_full,url_resized,resized_width_px,resized_height_px,manifest_url,canvas_id
0,1,JJ200,Paris_Archives_Nationales_JJ200_1.jpg,JJ200,1,garde supérieure,garde supérieure,4257,4215,https://iiif.irht.cnrs.fr/iiif/ark:/63955/v7xa53s0mymd/full/full/0/default.jpg,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/v7xa53s0mymd/full/,1200/0/default.jpg",1212,1200,https://api.irht.cnrs.fr/ark:/63955/frpbm744s3r5/manifest.json,https://arca.irht.cnrs.fr/iiif/128501/canvas/canvas-4290330
1,2,JJ200,Paris_Archives_Nationales_JJ200_2.jpg,JJ200,2,contre garde supérieure,contre garde supérieure,4257,4209,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vqq1lebcoo3i/full/full/0/default.jpg,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vqq1lebcoo3i/full/,1200/0/default.jpg",1214,1200,https://api.irht.cnrs.fr/ark:/63955/frpbm744s3r5/manifest.json,https://arca.irht.cnrs.fr/iiif/128501/canvas/canvas-4290331
2,3,JJ200,Paris_Archives_Nationales_JJ200_3.jpg,JJ200,3,1r,1r,3801,4503,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vlnsr0equbys/full/full/0/default.jpg,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vlnsr0equbys/full/1200,/0/default.jpg",1200,1422,https://api.irht.cnrs.fr/ark:/63955/frpbm744s3r5/manifest.json,https://arca.irht.cnrs.fr/iiif/128501/canvas/canvas-4290332


## Table `actes.csv`

In [45]:
%%time 

skipped = {}

with warnings.catch_warnings(record=True) as w_actes:
    warnings.simplefilter("always")
    df_actes_raw = pd.read_excel(INPUT_ACTES)
    skipped['actes'] = len(w_actes)



print(f"Actes  chargés  : {len(df_actes_raw):>6} lignes  ({skipped['actes']} ligne(s) ignorée(s))")


df_actes = df_actes_raw.copy()

df_actes.rename(columns={
    'ID-temporaire':        'acte_id',
    'Register':             'register',
    'Act_number':           'act_number',
    'Nvelle numérotation':  'new_numbering',
    'Folio Number ou page': 'folio_raw',
    'vérif':                'verified',
    'Note':                 'note',
}, inplace=True)



df_actes = df_actes[df_actes['register'].isin(REGISTRES_CIBLES)].copy()
print(f"Actes après filtre JJ96-JJ99 : {len(df_actes)} lignes")
print(df_actes['register'].value_counts().sort_index())

df_actes['folio_norm'] = df_actes['folio_raw'].apply(normalize_folio)


df_actes['folio_norm'] = df_actes['folio_raw'].apply(normalize_folio)


# Table de correspondance folio_norm → folio_sort_key par registre
# On garde une seule valeur de folio_sort_key par (register, folio_norm) via first()
folio_to_sortkey = (df_images[['register', 'folio_norm', 'folio_sort_key']]
                    .drop_duplicates(subset=['register', 'folio_norm'])
                    .reset_index(drop=True))

# Merge à la place du apply (plus rapide, pas de problème de multi-index)
df_actes = df_actes.merge(
    folio_to_sortkey,
    on=['register', 'folio_norm'],
    how='left'
)

# Signaler les folios non matchés
unmatched = df_actes[df_actes['folio_sort_key'].isna()]
if not unmatched.empty:
    print(f"⚠️  {len(unmatched)} acte(s) sans folio_sort_key :")
    print(unmatched[['acte_id', 'register', 'folio_norm']].to_string(index=False))
else:
    print("✅ Tous les folios ont été matchés.")
    


# Aplatir les concordances : chaque inventaire source donne 3 colonnes
concordance_cols = [c for c in df_actes.columns if '.xml_' in c]
inventory_sources = defaultdict(list)
for col in concordance_cols:
    src = col.split('.xml_')[0] + '.xml'
    inventory_sources[src].append(col)

for src, cols in inventory_sources.items():
    short = re.sub(r'^Paris_AN_JJ_inventaire_', '', src.replace('.xml', ''))
    short = short.replace('Guerin_tome1-tome12', 'Guerin')
    for suffix, key in [('_Act_number', 'act'), ('_Head', 'head'), ('_Locus', 'locus')]:
        col = next((c for c in cols if c.endswith(suffix)), None)
        if col:
            df_actes[f'conc_{short}_{key}'] = df_actes[col].fillna('')

BASE_COLS = ['acte_id', 'register', 'act_number', 'new_numbering',
             'folio_raw', 'folio_norm', 'verified', 'note']
CONC_COLS = [c for c in df_actes.columns if c.startswith('conc_')]
df_actes_out = df_actes[BASE_COLS + CONC_COLS].copy()

df_actes_out.to_csv(os.path.join(OUT_DIR, "actes.csv"), index=False, sep=SEP)
print(f"actes.csv → {len(df_actes_out)} lignes, {len(df_actes_out.columns)} colonnes")
df_actes_out.head(8)



Actes  chargés  :  39605 lignes  (0 ligne(s) ignorée(s))
Actes après filtre JJ96-JJ99 : 0 lignes
Series([], Name: count, dtype: int64)
✅ Tous les folios ont été matchés.
actes.csv → 0 lignes, 38 colonnes
CPU times: total: 4.86 s
Wall time: 4.97 s


,acte_id,register,act_number,new_numbering,folio_raw,folio_norm,verified,note,conc_Guerin_act,conc_Guerin_head,conc_Guerin_locus,conc_Longnon_act,conc_Longnon_head,conc_Longnon_locus,conc_Viard_act,conc_Viard_head,conc_Viard_locus,conc_Gascogne_act,conc_Gascogne_head,conc_Gascogne_locus,conc_Languedoc_act,conc_Languedoc_head,conc_Languedoc_locus,conc_Loire_act,conc_Loire_head,conc_Loire_locus,conc_Rouergue_act,conc_Rouergue_head,conc_Rouergue_locus,conc_IR421-JJA-JJ79A_act,conc_IR421-JJA-JJ79A_head,conc_IR421-JJA-JJ79A_locus,conc_IR422-JJ80-155_act,conc_IR422-JJ80-155_head,conc_IR422-JJ80-155_locus,conc_IR423-JJ156-211_act,conc_IR423-JJ156-211_head,conc_IR423-JJ156-211_locus


## Table `zones.csv`

In [46]:
skipped = {}

with warnings.catch_warnings(record=True) as w_zones:
    warnings.simplefilter("always")
    df_zones_raw = pd.read_csv(INPUT_ZONES, sep=',', dtype=str,
                               low_memory=False, on_bad_lines='warn')
    skipped['zones'] = len(w_zones)


def parse_labelstudio_rects(label_str):
    if pd.isna(label_str):
        return []

    if not str(label_str).strip():
        return []

    try:
        data = json.loads(label_str)
    except Exception:
        try:
            data = ast.literal_eval(label_str)
        except Exception:
            return []

    if not isinstance(data, list):
        return []

    rows = []

    for r in data:
        if 'rectanglelabels' not in r:
            continue

        rows.append({
            'class_name': r['rectanglelabels'][0],
            'x_pct': r['x'],
            'y_pct': r['y'],
            'w_pct': r['width'],
            'h_pct': r['height'],
            'image_width_px': r['original_width'],
            'image_height_px': r['original_height'],
        })

    return rows

# Supprime lignes entièrement vides ou sans annotation
df_zones_raw = df_zones_raw.dropna(how='all')
df_zones_raw = df_zones_raw.dropna(subset=['label'])
df_zones_raw = df_zones_raw[
    df_zones_raw['label'].astype(str).str.strip() != ''
]
print(f"Zones importées : {len(df_zones_raw)}")
print(f"Zones importées : {df_zones_raw.head(2)}")

df_zones_raw['url_image_full'] = df_zones_raw['image']    
# image_path contient des URLs IIIF, pas des chemins de fichiers :
# parse_image_stem ne s'applique pas ici.
# On utilise directement les colonnes 'registre' et 'ordre' déjà présentes dans df_zones_raw.

df_zones_raw['volume'] = df_zones_raw['volume'].apply(
    lambda r: re.sub(r'JJ(\d+)', lambda m: f"JJ{int(m.group(1)):03d}", str(r))
)
df_zones_raw['folio_sort_key'] = pd.to_numeric(df_zones_raw['image_id'], errors='coerce')


print(df_zones_raw)

rows = []
for _, r in df_zones_raw.iterrows():
    annotations = parse_labelstudio_rects(r['label'])
    for ann in annotations:
        rows.append({
            'url_image_full':   r['url_image_full'],
            'image_path':       r['image_path'],
            'image_filename':   r['image'],          # l'URL 1200, sert de clé de jointure
            'volume':           r['volume'],
            'folio_sort_key':   r['folio_sort_key'],
            'class_name':       ann['class_name'],
            'x_pct':            ann['x_pct'],
            'y_pct':            ann['y_pct'],
            'w_pct':            ann['w_pct'],
            'h_pct':            ann['h_pct'],
            'image_width_px':   ann['image_width_px'],
            'image_height_px':  ann['image_height_px'],
        })


df_zones = pd.DataFrame(rows)

df_zones['abs_x'] = safe_to_int(df_zones['x_pct'] / 100.0 * df_zones['image_width_px'])
df_zones['abs_y'] = safe_to_int(df_zones['y_pct'] / 100.0 * df_zones['image_height_px'])
df_zones['abs_w'] = safe_to_int(df_zones['w_pct'] / 100.0 * df_zones['image_width_px'])
df_zones['abs_h'] = safe_to_int(df_zones['h_pct'] / 100.0 * df_zones['image_height_px'])

df_zones = df_zones.sort_values(
    by=['volume', 'folio_sort_key', 'abs_y', 'abs_x'],
    ascending=[True, True, True, True]
    )

img_lookup = df_images[['image_id', 'image_filename', 'register',
                         'folio_label', 'folio_norm']].copy()

df_zones = df_zones.merge(img_lookup, on='image_filename', how='left')
df_zones['zone_id'] = np.arange(1, len(df_zones) + 1)


df_zones.rename(columns={
    'registre': 'register',
    'image': 'url_image_full',
}, inplace=True)


CLASS_MAP = {

    'AC': 0,
    'AI': 1,
    'AF': 2,
    'AM': 3,
    'NIA': 4,
    'Table': 5,
}

df_zones['class_id'] = df_zones['class_name'].map(CLASS_MAP)

ZONES_COLS = [
    'zone_id', 'image_id', 'volume', 'folio_sort_key',
    'folio_label', 'folio_norm',
    'class_id', 'class_name', 
    'abs_x', 'abs_y', 'abs_w', 'abs_h',
    'url_image_full', 'image_filename',
    'image_width_px', 'image_height_px',
]
df_zones = df_zones[ZONES_COLS]

df_zones.to_csv(os.path.join(OUT_DIR, "zones.csv"), index=False, sep=SEP)

print(f"zones.csv → {len(df_zones)} lignes, {len(df_zones.columns)} colonnes")
df_zones.head(10)

# %%
# Index image_id → liste de zones triées par abs_y (ordre de lecture)
zones_by_image = defaultdict(list)

for _, row in df_zones.iterrows():
    zones_by_image[row['image_id']].append(row.to_dict())

for iid in zones_by_image:
    zones_by_image[iid].sort(key=lambda r: r['abs_y'] if pd.notna(r['abs_y']) else 0)
    
# Ensemble de tous les zone_id (pour le rapport de couverture)
all_zone_ids      = set(df_zones['zone_id'].tolist())
attributed_zone_ids = set()   # alimenté lors de la construction des liaisons

print(f"Index zones_by_image  : {len(zones_by_image)} images ont des zones")
print(f"Total zones           : {len(all_zone_ids)}")


df_zones.head(20)


Zones importées : 3918
Zones importées :   annotation_id annotator                   created_at  \
1         48988         1  2026-06-03T14:47:04.301018Z   
2         48989         1  2026-06-03T14:47:04.301018Z   

               folio_label folio_sort_key     id  \
1  contre garde supérieure              2  48988   
2                       1r              3  48989   

                                                                             image  \
1  https://iiif.irht.cnrs.fr/iiif/ark:/63955/vqq1lebcoo3i/full/,1200/0/default.jpg   
2  https://iiif.irht.cnrs.fr/iiif/ark:/63955/vlnsr0equbys/full/1200,/0/default.jpg   

  image_id                             image_path  \
1        2  Paris_Archives_Nationales_JJ200_2.jpg   
2        3  Paris_Archives_Nationales_JJ200_3.jpg   

                                                                                                                     label  \
1  [{"x":-0.005,"y":0.0,"width":95.43,"height":100.0,"rotation":0,"rectanglelabels

,zone_id,image_id,volume,folio_sort_key,folio_label,folio_norm,class_id,class_name,abs_x,abs_y,abs_w,abs_h,url_image_full,image_filename,image_width_px,image_height_px
0,1,NaN,JJ200,2,NaN,NaN,2,AF,0,0,1159,1200,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vqq1lebcoo3i/full/,1200/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vqq1lebcoo3i/full/,1200/0/default.jpg",1214,1200
1,2,NaN,JJ200,3,NaN,NaN,3,AM,6,3,1194,1419,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vlnsr0equbys/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vlnsr0equbys/full/1200,/0/default.jpg",1200,1422
2,3,NaN,JJ200,4,NaN,NaN,2,AF,1,0,1197,502,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vm3ve3icqw2h/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vm3ve3icqw2h/full/1200,/0/default.jpg",1200,1355
3,4,NaN,JJ200,4,NaN,NaN,1,AI,0,502,1198,846,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vm3ve3icqw2h/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vm3ve3icqw2h/full/1200,/0/default.jpg",1200,1355
4,5,NaN,JJ200,5,NaN,NaN,2,AF,2,1,1198,1458,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxnv80kx42jr/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxnv80kx42jr/full/1200,/0/default.jpg",1200,1460
5,6,NaN,JJ200,6,NaN,NaN,2,AF,2,1,1198,1328,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vvbr91mlze72/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vvbr91mlze72/full/1200,/0/default.jpg",1200,1369
6,7,NaN,JJ200,6,NaN,NaN,3,AM,3,2,1197,1113,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vvbr91mlze72/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vvbr91mlze72/full/1200,/0/default.jpg",1200,1369
7,8,NaN,JJ200,6,NaN,NaN,1,AI,0,1113,1200,254,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vvbr91mlze72/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vvbr91mlze72/full/1200,/0/default.jpg",1200,1369
8,9,NaN,JJ200,7,NaN,NaN,2,AF,0,1,1198,713,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vam6syxo1fmp/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vam6syxo1fmp/full/1200,/0/default.jpg",1200,1458
9,10,NaN,JJ200,7,NaN,NaN,1,AI,0,713,1198,744,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vam6syxo1fmp/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vam6syxo1fmp/full/1200,/0/default.jpg",1200,1458


# Correction des données

## Pages non annotées

In [47]:
def find_missing_folios(df):
    missing = []

    for vol, g in df.groupby('volume'):
        folios = sorted(g['folio_sort_key'].unique())
        expected = set(range(min(folios), max(folios) + 1))
        actual = set(folios)
        gaps = sorted(expected - actual)

        if gaps:
            missing.append({
                'volume': vol,
                'missing_folios': gaps,
                'count': len(gaps)
            })

    return pd.DataFrame(missing)


df_missing = find_missing_folios(df_zones)
print(f"folios sans annotation : {len(df_missing)}")
df_missing.head(20)

folios sans annotation : 10


,volume,missing_folios,count
0,JJ201,"[232, 397, 554]",3
1,JJ202,"[566, 568, 572, 576, 578]",5
2,JJ203,"[706, 707]",2
3,JJ204,[811],1
4,JJ205,"[1054, 1242, 1592, 1601]",4
5,JJ207,[2163],1
6,JJ208,[2517],1
7,JJ209,"[2829, 3176]",2
8,JJ210,[3182],1
9,JJ211,[3607],1


## Cohérence des zones

In [48]:
def get_sequence(g):
    return g.sort_values(['abs_y', 'abs_x'])['class_name'].tolist()

def validate_sequence(seq):
    i = 0
    n = len(seq)
    if all(x == 'AC' for x in seq):
        return True
    
    if seq == ['AM']:
        return True

    if seq == ['NIA']:
        return True
    
    if seq == ['Table']:
        return True
    # --------
    # CAS 3 : (AF)? + AC* + (AI)?
    # --------
    state = "AF"

    # AF optionnel
    if i < n and seq[i] == 'AF':
        i += 1

    state = "AC"

    # AC*
    while i < n and seq[i] == 'AC':
        i += 1

    state = "AI"

    # AI optionnel
    if i < n and seq[i] == 'AI':
        i += 1

    # doit être consommé entièrement
    return i == n


def find_invalid_groups(df):
    bad = []

    for (vol, folio), g in df.groupby(['volume', 'folio_sort_key']):
        seq = get_sequence(g)

        if not validate_sequence(seq):
            bad.append({
                'volume': vol,
                'folio_sort_key': folio,
                'sequence': seq
            })

    return pd.DataFrame(bad)


df_invalid = find_invalid_groups(df_zones)

print(f"Groupes invalides : {len(df_invalid)}")
df_invalid.head(99)

Groupes invalides : 336


,volume,folio_sort_key,sequence
0,JJ200,6,"[AF, AM, AI]"
1,JJ200,39,"[AF, AM]"
2,JJ200,46,"[AC, AI, AC]"
3,JJ200,66,"[AC, AI, AC]"
4,JJ200,99,"[AI, AM]"
5,JJ200,108,"[AI, AC]"
6,JJ200,110,"[AC, AF]"
7,JJ200,114,"[AF, AI, AC]"
8,JJ200,133,"[AI, AC]"
9,JJ200,135,"[AC, AF, AC]"


## Cohérence transitions

In [49]:
def get_ordered_folios(df):
    """Retourne les folios ordonnés par volume et folio_sort_key."""
    return (
        df[['volume', 'folio_sort_key']]
        .drop_duplicates()
        .sort_values(['volume', 'folio_sort_key'])
        .values.tolist()
    )

def get_first_last_zones(df, vol, folio):
    """Retourne la première et dernière zone d'un folio, triées du haut vers le bas (abs_y puis abs_x)."""
    g = df[(df['volume'] == vol) & (df['folio_sort_key'] == folio)]
    seq = get_sequence(g)  # déjà trié par abs_y, abs_x dans get_sequence()
    if not seq:
        return None, None
    return seq[0], seq[-1]

# Règles : dernière zone → classes autorisées en première zone de la page suivante
TRANSITION_RULES = {
    'AC': {'AI', 'AC', 'NIA'},
    'AM': {'AM', 'AF'},
    'AI': {'AM', 'AF'},
    'AF': {'AI', 'AC', 'NIA'},
}

def find_invalid_transitions(df):
    folios = get_ordered_folios(df)
    # print(folios[1000:1010])
    bad = []

    for idx in range(len(folios) - 1):
        vol_cur,  folio_cur  = folios[idx]
        vol_next, folio_next = folios[idx + 1]

        # On ne contrôle les transitions qu'au sein d'un même volume
        if vol_cur != vol_next:
            continue

        _, last_zone  = get_first_last_zones(df, vol_cur,  folio_cur)
        first_zone, _ = get_first_last_zones(df, vol_next, folio_next)

        if last_zone is None or first_zone is None:
            continue

        allowed = TRANSITION_RULES.get(last_zone)

        # Pas de règle définie pour cette dernière zone → on ignore
        if allowed is None:
            continue

        if first_zone not in allowed:
            bad.append({
                'volume':          vol_cur,
                'folio_cur':       folio_cur,
                'folio_next':      folio_next,
                'last_zone_cur':   last_zone,
                'first_zone_next': first_zone,
                'allowed':         sorted(allowed),
            })

    return pd.DataFrame(bad)


df_invalid_transitions = find_invalid_transitions(df_zones)
print(f"Transitions invalides : {len(df_invalid_transitions)}")
df_invalid_transitions.head(99)


Transitions invalides : 658


,volume,folio_cur,folio_next,last_zone_cur,first_zone_next,allowed
0,JJ200,2,3,AF,AM,"[AC, AI, NIA]"
1,JJ200,5,6,AF,AF,"[AC, AI, NIA]"
2,JJ200,13,14,AF,AF,"[AC, AI, NIA]"
3,JJ200,19,20,AC,AM,"[AC, AI, NIA]"
4,JJ200,22,23,AC,AF,"[AC, AI, NIA]"
5,JJ200,26,27,AC,AF,"[AC, AI, NIA]"
6,JJ200,27,28,AC,AF,"[AC, AI, NIA]"
7,JJ200,28,29,AF,AF,"[AC, AI, NIA]"
8,JJ200,31,32,AC,AF,"[AC, AI, NIA]"
9,JJ200,33,34,AI,AC,"[AF, AM]"


# Fusion des données

## Table `actes_images_zones.csv`

### Règles de liaison
- La **première zone** de chaque acte doit être **AI** ou **AC**,
  sur l'image dont le `folio_norm` correspond au folio déclaré dans le tableau des actes.
- Après une zone **AI**, on cherche séquentiellement :
  - des zones **AM** (pages entières intermédiaires, une par image)
   - puis une zone **AF** (fin de l'acte, première zone en haut d'une nouvelle page)
  - Une zone **AC** clôt immédiatement la liaison.

In [50]:

# Logs pour le rapport de qualité
log_actes_sans_folio          = []
log_actes_sans_image          = []
log_actes_sans_zone_initiale  = []
log_actes_structure_invalide  = []
log_zones_orphelines          = []

# %%
def get_next_images(image_id, register):
    """
    Renvoie les image_id du même registre qui suivent image_id,
    dans l'ordre séquentiel du fichier images source.
    """
    reg_imgs = images_by_register.get(register, [])
    try:
        idx = reg_imgs.index(image_id)
        return reg_imgs[idx + 1:]
    except ValueError:
        return []


def make_liaison(acte_id, act_number, register, folio_norm,
                 image_id, img_order, zone, role, zone_order_global,
                 statut_zone, statut_structure=None):
    """Construit un enregistrement de liaison (dict)."""
    return {
        'acte_id':             acte_id,
        'act_number':          act_number,
        'register':            register,
        'folio_norm':          folio_norm,
        'image_id':            image_id,
        'image_order':         img_order,
        'zone_id':             zone['zone_id'] if zone else None,
        'class_id':            zone['class_id'] if zone else None,
        'class_name':          zone['class_name'] if zone else None,
        'role':                role,
        'zone_order_in_image': (int(zone['abs_y'])
                                if zone and pd.notna(zone.get('abs_y')) else None),
        'zone_order_global':   zone_order_global,
        'abs_x':               zone['abs_x'] if zone else None,
        'abs_y':               zone['abs_y'] if zone else None,
        'abs_w':               zone['abs_w'] if zone else None,
        'abs_h':               zone['abs_h'] if zone else None,
        'statut_zone':         statut_zone,
        'statut_structure':    statut_structure,
    }


def build_liaisons_for_acte(acte):
    """
    Construit la liste des enregistrements de liaison pour un acte.
    Retourne (list[dict], statut_structure).
    """
    liaisons   = []
    register   = acte['register']
    folio_norm = acte['folio_norm']
    acte_id    = acte['acte_id']
    act_number = acte['act_number']

    # A — folio et registre valides
    if not folio_norm or not register or folio_norm == 'nan':
        log_actes_sans_folio.append(
            {'acte_id': acte_id, 'act_number': act_number})
        return [], 'ignoré_sans_folio'

    # B — image d'ancrage
    volume       = acte['volume']
    sort_key     = acte['folio_sort_key']
    images_ancrage = img_by_folio.get((volume, sort_key), []) if pd.notna(sort_key) else []
    

    if not images_ancrage:
        log_actes_sans_image.append({
            'acte_id': acte_id, 'act_number': act_number,
            'register': register, 'folio_norm': folio_norm})
        return [], 'avertissement_sans_image'

    roles_trouves     = []
    zone_order_global = 0

    for image_id in images_ancrage:
        zones_image = zones_by_image.get(image_id, [])

        # C — zone initiale : première AI ou AC dans l'image (ordre abs_y)
        zone_initiale = next(
            (z for z in zones_image 
             if z['class_name'] in ('AI', 'AC')
             and z['zone_id'] not in attributed_zone_ids), None)
                    
        if zone_initiale is None:
            log_actes_sans_zone_initiale.append({
                'acte_id': acte_id, 'act_number': act_number,
                'register': register, 'folio_norm': folio_norm,
                'image_id': image_id})
            liaisons.append(make_liaison(
                acte_id, act_number, register, folio_norm,
                image_id, img_order=1, zone=None, role=None,
                zone_order_global=None,
                statut_zone='ancrage_sans_zone_initiale'))
            return liaisons, 'avertissement_sans_zone_initiale'

        zone_order_global += 1
        attributed_zone_ids.add(zone_initiale['zone_id'])

        # D — cas AC : acte complet, terminé
        if zone_initiale['class_name'] == 'AC':
            roles_trouves.append('AC')
            liaisons.append(make_liaison(
                acte_id, act_number, register, folio_norm,
                image_id, img_order=1, zone=zone_initiale, role='AC',
                zone_order_global=zone_order_global,
                statut_zone='valide'))
            return liaisons, 'valide_AC'

        # D — cas AI : chercher la suite
        roles_trouves.append('AI')
        liaisons.append(make_liaison(
            acte_id, act_number, register, folio_norm,
            image_id, img_order=1, zone=zone_initiale, role='AI',
            zone_order_global=zone_order_global,
            statut_zone='valide'))

        # E — images suivantes : AM* puis AF
        img_order = 2
        for next_img_id in get_next_images(image_id, register):
            zones_suiv = zones_by_image.get(next_img_id, [])
            zones_am = [z for z in zones_suiv
                        if z['class_name'] == 'AM' and z['zone_id'] not in attributed_zone_ids]
            zones_af = [z for z in zones_suiv
                        if z['class_name'] == 'AF' and z['zone_id'] not in attributed_zone_ids]

            if zones_am and not zones_af:
                # Page entièrement couverte par l'acte (AM)
                for z_am in zones_am:   # une seule AM par image en théorie
                    zone_order_global += 1
                    attributed_zone_ids.add(z_am['zone_id'])
                    roles_trouves.append('AM')
                    liaisons.append(make_liaison(
                        acte_id, act_number, register, folio_norm,
                        next_img_id, img_order=img_order, zone=z_am, role='AM',
                        zone_order_global=zone_order_global,
                        statut_zone='valide'))
                img_order += 1

            elif zones_af:
                # Page de fin : première AF (haut de page = abs_y minimal)
                z_af = zones_af[0]
                zone_order_global += 1
                attributed_zone_ids.add(z_af['zone_id'])
                roles_trouves.append('AF')
                liaisons.append(make_liaison(
                    acte_id, act_number, register, folio_norm,
                    next_img_id, img_order=img_order, zone=z_af, role='AF',
                    zone_order_global=zone_order_global,
                    statut_zone='valide'))
                break   # AF = acte terminé

            else:
                # Ni AM ni AF : suite manquante
                liaisons.append(make_liaison(
                    acte_id, act_number, register, folio_norm,
                    next_img_id, img_order=img_order, zone=None, role=None,
                    zone_order_global=None,
                    statut_zone='zone_suite_manquante'))
                roles_trouves.append('?')
                break

    # F — valider la structure de l'acte
    if roles_trouves == ['AC']:
        statut = 'valide_AC'
    elif (roles_trouves
          and roles_trouves[0] == 'AI'
          and roles_trouves[-1] == 'AF'
          and all(r in ('AI', 'AM', 'AF') for r in roles_trouves)):
        statut = 'valide_AI_AM_AF'
    else:
        statut = 'structure_invalide'
        log_actes_structure_invalide.append({
            'acte_id':       acte_id,
            'act_number':    act_number,
            'roles_trouves': ', '.join(roles_trouves)})

    for l in liaisons:
        l['statut_structure'] = statut

    return liaisons, statut

# %%
# Détection des zones orphelines (image non trouvée lors de la jointure)
orphelines = df_zones[df_zones['image_id'].isna()]
for _, row in orphelines.iterrows():
    log_zones_orphelines.append({
        'zone_id':        row['zone_id'],
        'image_filename': row['image_filename'],
        'class_name':     row['class_name'],
    })

# %%
# Construction de toutes les liaisons
all_liaisons = []
for _, acte in df_actes.iterrows():
    liaisons, _ = build_liaisons_for_acte(acte)
    all_liaisons.extend(liaisons)

df_liaison = pd.DataFrame(all_liaisons)
if not df_liaison.empty:
    df_liaison.insert(0, 'liaison_id', range(1, len(df_liaison) + 1))

df_liaison.to_csv(os.path.join(OUT_DIR, "actes_images_zones.csv"),
                  index=False, sep=SEP)
print(f"actes_images_zones.csv → {len(df_liaison)} lignes, "
      f"{len(df_liaison.columns)} colonnes")
df_liaison.head(5)



actes_images_zones.csv → 0 lignes, 0 colonnes


""


## Rapport de qualité

In [51]:

print("=" * 60)
print("A. CHARGEMENT")
print("=" * 60)
print(f"  Images chargées : {len(df_images_raw):>6}")
print(f"  Zones  chargées : {len(df_zones_raw):>6}")
print(f"  Actes  chargés  : {len(df_actes_raw):>6}")
print()
for fichier, n in skipped.items():
    if n:
        print(f"  ⚠ {n} ligne(s) ignorée(s) au parsing dans : {fichier}")


# %% [markdown]
# ### B — Problèmes de jointure et de localisation

# %%
print("=" * 60)
print("B. JOINTURES ET LOCALISATION")
print("=" * 60)

print(f"\n  Zones orphelines (image non trouvée) : {len(log_zones_orphelines)}")
if log_zones_orphelines:
    print(pd.DataFrame(log_zones_orphelines).to_string(index=False))

print(f"\n  Actes sans folio/registre (ignorés)  : {len(log_actes_sans_folio)}")
if log_actes_sans_folio:
    print(pd.DataFrame(log_actes_sans_folio).to_string(index=False))

print(f"\n  Actes sans image d'ancrage           : {len(log_actes_sans_image)}")
if log_actes_sans_image:
    print(pd.DataFrame(log_actes_sans_image).to_string(index=False))

print(f"\n  Actes sans zone AI/AC sur l'ancrage  : {len(log_actes_sans_zone_initiale)}")
if log_actes_sans_zone_initiale:
    print(pd.DataFrame(log_actes_sans_zone_initiale).to_string(index=False))


# %% [markdown]
# ### C — Validation des structures d'actes

# %%
print("=" * 60)
print("C. STRUCTURES D'ACTES")
print("=" * 60)

if not df_liaison.empty:
    statuts = (df_liaison.drop_duplicates('acte_id')
                         .groupby('statut_structure', dropna=False)
                         .size()
                         .reset_index(name='nb_actes'))
    print(statuts.to_string(index=False))
else:
    print("  Aucune liaison produite.")

print(f"\n  Actes à structure invalide : {len(log_actes_structure_invalide)}")
if log_actes_structure_invalide:
    print(pd.DataFrame(log_actes_structure_invalide).to_string(index=False))


# %% [markdown]
# ### D — Couverture des zones

# %%
print("=" * 60)
print("D. COUVERTURE DES ZONES")
print("=" * 60)

# Inventaire de tous les labels présents
print("\n  Labels de zones présents dans le fichier source :")
labels_counts = (df_zones.groupby('class_name', dropna=False)
                          .size()
                          .reset_index(name='nb_zones'))
labels_counts['type'] = labels_counts['class_name'].apply(
    lambda x: 'attendu' if x in LABELS_ATTENDUS else '⚠ inattendu')
print(labels_counts.to_string(index=False))

# %%
# Zones attendues (AC/AI/AM/AF) vs inattendues
LABELS_ATTENDUS  = {'AC', 'AI', 'AM', 'AF'}
LABELS_HORS_SCOPE = {'NIA', 'Table'}          # dans les paramètres en haut

df_zones_attendues   = df_zones[df_zones['class_name'].isin(LABELS_ATTENDUS)]
df_zones_hors_scope  = df_zones[df_zones['class_name'].isin(LABELS_HORS_SCOPE)]
df_zones_inattendues = df_zones[~df_zones['class_name'].isin(LABELS_ATTENDUS | LABELS_HORS_SCOPE)]



nb_att        = len(df_zones_attendues)
nb_hors_scope = len(df_zones_hors_scope)
nb_inatt      = len(df_zones_inattendues)
nb_attr       = len(df_zones_attendues[df_zones_attendues['zone_id'].isin(attributed_zone_ids)])

print(f"  Zones de label attendu   (AC/AI/AM/AF) : {nb_att}")
print(f"    → attribuées à un acte valide        : {nb_attr}")
print(f"    → NON attribuées (anomalie)           : {nb_att - nb_attr}")
print(f"\n  Zones hors scope (NIA/Table, normal)   : {nb_hors_scope}")
print(f"\n  Zones de label vraiment inattendu      : {nb_inatt}")

# %%
# Détail des zones attendues non attribuées
zones_attendues_non_attr = df_zones_attendues[
    ~df_zones_attendues['zone_id'].isin(attributed_zone_ids)
].copy()

if not zones_attendues_non_attr.empty:
    # Identifier l'acte candidat : un acte dont (register, folio_norm) matche la zone
    actes_idx = df_actes.set_index(['register', 'folio_norm'])['acte_id'].to_dict()

    def acte_candidat(row):
        key = (row['register'], row['folio_norm'])
        return actes_idx.get(key, 'aucun')

    zones_attendues_non_attr['acte_candidat'] = zones_attendues_non_attr.apply(
        acte_candidat, axis=1)

    # Cause probable
    def cause(row):
        if row['acte_candidat'] == 'aucun':
            return 'aucun acte ne pointe vers ce folio'
        statut_acte = None
        if not df_liaison.empty and 'acte_id' in df_liaison.columns:
            rows = df_liaison[df_liaison['acte_id'] == str(row['acte_candidat'])]
            if not rows.empty:
                statut_acte = rows.iloc[0]['statut_structure']
        if statut_acte == 'structure_invalide':
            return 'acte candidat à structure invalide'
        return 'doublon ou zone hors séquence attendue'

    zones_attendues_non_attr['cause_probable'] = zones_attendues_non_attr.apply(
        cause, axis=1)

    cols_rapport = ['zone_id', 'image_id', 'class_name', 'folio_norm',
                    'register', 'acte_candidat', 'cause_probable']
    print("\n  ⚠ Zones attendues non attribuées (anomalies à corriger) :", len(zones_attendues_non_attr[cols_rapport]))
    print(zones_attendues_non_attr[cols_rapport][0:10].to_string(index=False))

# %%
# Détail des zones inattendues
if not df_zones_inattendues.empty:
    cols_inatt = ['zone_id', 'image_id', 'class_name', 'folio_norm',
                  'register', 'folio_sort_key', 'url_image_full']
    print("\n  ℹ Zones de label inattendu (hors scope du modèle d'acte) :")
    print(df_zones_inattendues[cols_inatt].to_string(index=False))


# %% [markdown]
# ### E — Résumé exécutif

# %%
print("=" * 60)
print("E. RÉSUMÉ EXÉCUTIF")
print("=" * 60)

total_actes = len(df_actes)
actes_valides = 0
if not df_liaison.empty:
    actes_valides = df_liaison[
        df_liaison['statut_structure'].isin(['valide_AC', 'valide_AI_AM_AF'])
    ]['acte_id'].nunique()

taux_actes = actes_valides / total_actes * 100 if total_actes else 0
taux_zones = nb_attr / nb_att * 100 if nb_att else 0

nb_anomalies = (len(log_zones_orphelines)
                + len(log_actes_sans_image)
                + len(log_actes_sans_zone_initiale)
                + len(log_actes_structure_invalide)
                + (nb_att - nb_attr))

print(f"\n  Taux de couverture des actes  : {actes_valides}/{total_actes} "
      f"({taux_actes:.1f}%)")
print(f"  Taux de couverture des zones  : {nb_attr}/{nb_att} "
      f"({taux_zones:.1f}%) [labels AC/AI/AM/AF uniquement]")
print(f"\n  Total anomalies à traiter     : {nb_anomalies}")
print(f"    dont zones orphelines        : {len(log_zones_orphelines)}")
print(f"    dont actes sans image        : {len(log_actes_sans_image)}")
print(f"    dont actes sans zone initiale: {len(log_actes_sans_zone_initiale)}")
print(f"    dont structures invalides    : {len(log_actes_structure_invalide)}")
print(f"    dont zones attendues non attr: {nb_att - nb_attr}")
print()
print(f"  Fichiers produits dans : {os.path.abspath(OUT_DIR)}/")
print(f"    images.csv              ({len(df_images)} lignes)")
print(f"    zones.csv               ({len(df_zones)} lignes)")
print(f"    actes.csv               ({len(df_actes_out)} lignes)")
print(f"    actes_images_zones.csv  ({len(df_liaison)} lignes)")

A. CHARGEMENT
  Images chargées :   3950
  Zones  chargées :   3918
  Actes  chargés  :  39605

B. JOINTURES ET LOCALISATION

  Zones orphelines (image non trouvée) : 7637
 zone_id                                                                  image_filename class_name
       1 https://iiif.irht.cnrs.fr/iiif/ark:/63955/vqq1lebcoo3i/full/,1200/0/default.jpg         AF
       2 https://iiif.irht.cnrs.fr/iiif/ark:/63955/vlnsr0equbys/full/1200,/0/default.jpg         AM
       3 https://iiif.irht.cnrs.fr/iiif/ark:/63955/vm3ve3icqw2h/full/1200,/0/default.jpg         AF
       4 https://iiif.irht.cnrs.fr/iiif/ark:/63955/vm3ve3icqw2h/full/1200,/0/default.jpg         AI
       5 https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxnv80kx42jr/full/1200,/0/default.jpg         AF
       6 https://iiif.irht.cnrs.fr/iiif/ark:/63955/vvbr91mlze72/full/1200,/0/default.jpg         AF
       7 https://iiif.irht.cnrs.fr/iiif/ark:/63955/vvbr91mlze72/full/1200,/0/default.jpg         AM
       8 https://iiif.irht.c

KeyError: 'register'

# from csv to LabelStudio json

In [ ]:
import pandas as pd
import json
import sys
from collections import defaultdict

# Lecture du CSV (séparateur tabulation)

df = df_zones_raw

df["Confidence"] = pd.to_numeric(df["Confidence"], errors="coerce")
df["Image_Width"] = pd.to_numeric(df["Image_Width"], errors="coerce")
df["Image_Height"] = pd.to_numeric(df["Image_Height"], errors="coerce")
 
# Extraction registre + numéro d'ordre depuis Image_Path
# ex: .../Paris_Archives_Nationales_JJ096_100.jpg → registre=JJ096, ordre=100
def parse_image_stem(image_path):
    stem = image_path.rsplit("/", 1)[-1].rsplit(".", 1)[0]  # retire dossier et extension
    parts = stem.split("_")
    registre = parts[-2]                  # "JJ096"
    ordre = int(parts[-1])               # 100
    return registre, ordre
 
df[["Registre", "Ordre"]] = df["Image_Path"].apply(
    lambda p: pd.Series(parse_image_stem(p))
)
 
# Tri par registre puis numéro d'ordre
df = df.sort_values(["Registre", "Ordre"]).reset_index(drop=True)
 
# Regroupement par image (en conservant l'ordre du df trié)
seen = {}
ordered_keys = []
for path in df["Image_Path"]:
    if path not in seen:
        seen[path] = True
        ordered_keys.append(path)
 
grouped = df.groupby("Image_Path", sort=False)
tasks = []
 
for image_path in ordered_keys:
    group = grouped.get_group(image_path)
    row0 = group.iloc[0]
    img_w = int(row0["Image_Width"])
    img_h = int(row0["Image_Height"])
    url_image = row0["Url_Image"].replace("/full/full/", "/full/1200,/")
 
    annotations = []
    for _, det in group.iterrows():
        # Coordonnées YOLO normalisées : cx cy w h
        parts = str(det["Detected_coordinates"]).split()
        cx, cy, bw, bh = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])
 
        # Conversion en % (Label Studio : x,y coin supérieur gauche)
        x_pct = (cx - bw / 2) * 100
        y_pct = (cy - bh / 2) * 100
        w_pct = bw * 100
        h_pct = bh * 100
 
        annotations.append({
            "id": f"{det['Class_Id']}_{det['Class_Name']}_{round(float(det['Confidence']), 4)}",
            "type": "rectanglelabels",
            "value": {
                "x": round(x_pct, 4),
                "y": round(y_pct, 4),
                "width": round(w_pct, 4),
                "height": round(h_pct, 4),
                "rotation": 0,
                "rectanglelabels": [det["Class_Name"]]
            },
            "to_name": "image",
            "from_name": "label",
            "image_rotation": 0,
            "original_width": img_w,
            "original_height": img_h
        })
 
    task = {
        "data": {
            "image": url_image,
            "image_path": image_path,
            "registre": row0["Registre"],
            "ordre": int(row0["Ordre"]),
        },
        "annotations": [{"result": annotations}],
        "meta": {"source_file": image_path}
    }
    tasks.append(task)



output_path = "label_studio_import_JJ96-99.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(tasks, f, ensure_ascii=False, indent=2)


print(f"✓ {len(tasks)} tâche(s) générée(s) → {output_path}")